# Machine Learning Project: To bee or not to bee

---
## Part 1: Feature Extraction

This section contains the feature extraction code.
The pre-computed features are saved in `train_features.csv` and loaded directly in Part 2.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from skimage.measure import regionprops, label, perimeter

In [ ]:
TRAIN_DIR = Path("train")
IMG_DIR = TRAIN_DIR
MASK_DIR = TRAIN_DIR / "masks"
CLASSIF_PATH = TRAIN_DIR / "classif.xlsx"

In [ ]:
def load_image(path):
    return Image.open(path).convert("RGB")


def load_mask(path, image_size):
    mask = Image.open(path).convert("L")
    mask = mask.resize(image_size)
    mask = np.array(mask)
    return mask > 0

In [ ]:
def rgb_features(image, mask):
    """Min, max, mean, median, std for R, G, B within the bug mask."""
    image = np.array(image)
    pixels = image[mask]
    features = {}
    for i, color in enumerate(["R", "G", "B"]):
        values = pixels[:, i]
        features[f"{color}_min"] = np.min(values)
        features[f"{color}_max"] = np.max(values)
        features[f"{color}_mean"] = np.mean(values)
        features[f"{color}_median"] = np.median(values)
        features[f"{color}_std"] = np.std(values)
    return features


def bug_ratio(mask):
    """Ratio of bug pixels to total image pixels."""
    return np.sum(mask) / mask.size


def shape_symmetry(mask):
    """Horizontal shape symmetry of the mask."""
    flipped_mask = np.fliplr(mask)
    return np.mean(mask == flipped_mask)


def color_symmetry(image, mask):
    """Mean absolute color difference between left and right halves of the bug."""
    h, w, _ = image.shape
    mid = w // 2

    left_img = image[:, :mid]
    right_img = image[:, -mid:]
    right_img = np.fliplr(right_img)

    left_mask = mask[:, :mid]
    right_mask = mask[:, -mid:]
    right_mask = np.fliplr(right_mask)

    common_mask = left_mask & right_mask

    if np.sum(common_mask) == 0:
        return 0

    diff = np.abs(
        left_img[common_mask].astype(float)
        - right_img[common_mask].astype(float)
    )
    return np.mean(diff)


def extra_shape_features(mask):
    """Aspect ratio and circularity of the largest connected component."""
    features = {}
    labeled_mask = label(mask)
    props = regionprops(labeled_mask)

    if len(props) == 0:
        features["aspect_ratio"] = 0
        features["circularity"] = 0
        return features

    region = max(props, key=lambda r: r.area)
    min_row, min_col, max_row, max_col = region.bbox
    width = max_col - min_col
    height = max_row - min_row
    area = region.area
    perim = perimeter(mask)

    features["aspect_ratio"] = width / height if height != 0 else 0
    features["circularity"] = (
        4 * np.pi * area / (perim ** 2) if perim != 0 else 0
    )
    return features

In [ ]:
# Feature extraction loop
# Run this cell only if train_features.csv needs to be regenerated

REGENERATE_FEATURES = False

if REGENERATE_FEATURES:
    all_features = []
    skipped_images = []

    for i in range(1, 251):
        img_path = IMG_DIR / f"{i}.JPG"
        mask_path = MASK_DIR / f"binary_{i}.tif"

        if not img_path.exists() or not mask_path.exists():
            skipped_images.append(i)
            continue

        image = load_image(img_path)
        mask = load_mask(mask_path, image.size)
        image = np.array(image)

        row = {
            "ID": i,
            "bug_ratio": bug_ratio(mask),
            "shape_symmetry": shape_symmetry(mask),
            "color_symmetry": color_symmetry(image, mask),
        }
        row.update(rgb_features(image, mask))
        row.update(extra_shape_features(mask))
        all_features.append(row)

    features_df = pd.DataFrame(all_features)
    classif_df = pd.read_excel(CLASSIF_PATH)
    df = features_df.merge(classif_df, on="ID", how="left")
    df.to_csv("train_features.csv", index=False)
    print(f"Saved train_features.csv: {len(df)} samples, skipped: {skipped_images}")
else:
    print("Skipping feature extraction (loading from CSV in Part 2)")

---
## Part 2: Data Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Load pre-computed features
df = pd.read_csv("train_features.csv")

# Clean classes
df = df[df["bug type"] != "Dragonfly"].copy()
df["bug type"] = df["bug type"].replace("Bee & Bumblebee", "Bee")
df = df.reset_index(drop=True)

print(f"Dataset: {len(df)} samples, {df['bug type'].nunique()} classes")
print(df["bug type"].value_counts())

### 2.1 - Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bug type distribution
bug_counts = df["bug type"].value_counts()
sns.barplot(x=bug_counts.index, y=bug_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("Distribution of Bug Types")
axes[0].set_xlabel("Bug Type")
axes[0].set_ylabel("Count")
for i, v in enumerate(bug_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

# Species distribution (top 10)
species_counts = df["species"].value_counts().head(10)
sns.barplot(x=species_counts.values, y=species_counts.index, ax=axes[1], palette="mako")
axes[1].set_title("Top 10 Species")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Species")

plt.tight_layout()
plt.show()

In [ ]:
# Species composition within each bug type
ct = pd.crosstab(df["species"], df["bug type"])
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]

ct.T.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="tab20")
plt.title("Species Composition within Each Bug Type")
plt.xlabel("Bug Type")
plt.ylabel("Count")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

### 2.2 - PCA Projection (2D)

In [ ]:
# Prepare feature matrix
feature_cols = [c for c in df.columns if c not in ["ID", "bug type", "species"]]
X = df[feature_cols].values
y = df["bug type"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix: {X_scaled.shape} ({len(feature_cols)} features)")
print(f"Features: {feature_cols}")

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for bug_type in np.unique(y):
    mask = y == bug_type
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=bug_type, alpha=0.7, s=50)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA Projection (2D)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nTotal variance explained by 2 components: {sum(pca.explained_variance_ratio_)*100:.1f}%")
print(f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")

### 2.3 - t-SNE Projection (non-linear approach)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for bug_type in np.unique(y):
    mask = y == bug_type
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=bug_type, alpha=0.7, s=50)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE Projection (perplexity=30)")
plt.legend()
plt.tight_layout()
plt.show()

### 2.4 - UMAP Projection (non-linear approach too)

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
for bug_type in np.unique(y):
    mask = y == bug_type
    plt.scatter(X_umap[mask, 0], X_umap[mask, 1], label=bug_type, alpha=0.7, s=50)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP Projection (n_neighbors=15, min_dist=0.1)")
plt.legend()
plt.tight_layout()
plt.show()

### 2.5 - Comparison of Projections

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

projections = [
    (X_pca, "PCA"),
    (X_tsne, "t-SNE"),
    (X_umap, "UMAP"),
]

for ax, (X_proj, title) in zip(axes, projections):
    for bug_type in np.unique(y):
        mask = y == bug_type
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1], label=bug_type, alpha=0.7, s=40)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle("Comparison of Dimensionality Reduction Methods", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 3 - Machine Learning & Deep Learning

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    silhouette_score, adjusted_rand_score, normalized_mutual_info_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### 3.1 - Supervised Method 1: Support Vector Machine (SVM)

In [ ]:
# SVM with GridSearch
svm_params = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1],
    "kernel": ["rbf"],
}

svm_grid = GridSearchCV(
    SVC(random_state=42),
    svm_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
svm_grid.fit(X_scaled, y)

print("SVM — Best parameters:", svm_grid.best_params_)
print(f"SVM — Best F1 macro (CV): {svm_grid.best_score_:.4f}")

# Cross-validation scores with best model
svm_best = svm_grid.best_estimator_
svm_cv_scores = cross_val_score(svm_best, X_scaled, y, cv=cv, scoring="f1_macro")
print(f"SVM — F1 macro per fold: {svm_cv_scores}")
print(f"SVM — Mean F1 macro: {svm_cv_scores.mean():.4f} (+/- {svm_cv_scores.std():.4f})")

In [ ]:
# SVM — Classification report and confusion matrix
from sklearn.model_selection import cross_val_predict

y_pred_svm = cross_val_predict(svm_best, X_scaled, y, cv=cv)

print("SVM — Classification Report (CV predictions):\n")
print(classification_report(y, y_pred_svm))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y, y_pred_svm, ax=ax, cmap="Blues")
ax.set_title("SVM — Confusion Matrix (Cross-Validated)")
plt.tight_layout()
plt.show()

### 3.2 - Supervised Method 2: K-Nearest Neighbors (KNN)

In [ ]:
# KNN with GridSearch
knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
knn_grid.fit(X_scaled, y)

print("KNN — Best parameters:", knn_grid.best_params_)
print(f"KNN — Best F1 macro (CV): {knn_grid.best_score_:.4f}")

knn_best = knn_grid.best_estimator_
knn_cv_scores = cross_val_score(knn_best, X_scaled, y, cv=cv, scoring="f1_macro")
print(f"KNN — F1 macro per fold: {knn_cv_scores}")
print(f"KNN — Mean F1 macro: {knn_cv_scores.mean():.4f} (+/- {knn_cv_scores.std():.4f})")

In [ ]:
# KNN — Classification report and confusion matrix
y_pred_knn = cross_val_predict(knn_best, X_scaled, y, cv=cv)

print("KNN — Classification Report (CV predictions):\n")
print(classification_report(y, y_pred_knn))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y, y_pred_knn, ax=ax, cmap="Greens")
ax.set_title("KNN — Confusion Matrix (Cross-Validated)")
plt.tight_layout()
plt.show()

### 3.3 - Ensemble Method: Random Forest

In [ ]:
# Random Forest with GridSearch
rf_params = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5],
    "class_weight": ["balanced"],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
rf_grid.fit(X_scaled, y)

print("Random Forest — Best parameters:", rf_grid.best_params_)
print(f"Random Forest — Best F1 macro (CV): {rf_grid.best_score_:.4f}")

rf_best = rf_grid.best_estimator_
rf_cv_scores = cross_val_score(rf_best, X_scaled, y, cv=cv, scoring="f1_macro")
print(f"Random Forest — F1 macro per fold: {rf_cv_scores}")
print(f"Random Forest — Mean F1 macro: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std():.4f})")

In [ ]:
# Random Forest — Classification report and confusion matrix
y_pred_rf = cross_val_predict(rf_best, X_scaled, y, cv=cv)

print("Random Forest — Classification Report (CV predictions):\n")
print(classification_report(y, y_pred_rf))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y, y_pred_rf, ax=ax, cmap="Oranges")
ax.set_title("Random Forest — Confusion Matrix (Cross-Validated)")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances from Random Forest
importances = rf_best.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.bar(range(len(feature_cols)), importances[indices], color="darkorange")
plt.xticks(range(len(feature_cols)), [feature_cols[i] for i in indices], rotation=45, ha="right")
plt.title("Random Forest — Feature Importances")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

print("Top 5 features:")
for i in range(5):
    print(f"  {feature_cols[indices[i]]}: {importances[indices[i]]:.4f}")

### 3.4 - Clustering

In [ ]:
# K-Means clustering
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)
kmeans_ari = adjusted_rand_score(y_encoded, kmeans_labels)
kmeans_nmi = normalized_mutual_info_score(y_encoded, kmeans_labels)

print("K-Means (k=5):")
print(f"  Silhouette Score: {kmeans_silhouette:.4f}")
print(f"  Adjusted Rand Index (ARI): {kmeans_ari:.4f}")
print(f"  Normalized Mutual Information (NMI): {kmeans_nmi:.4f}")

# Visualize on PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(5):
    mask = kmeans_labels == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=f"Cluster {i}", alpha=0.7, s=50)
axes[0].set_title("K-Means Clusters (PCA projection)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend()

for bug_type in np.unique(y):
    mask = y == bug_type
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=bug_type, alpha=0.7, s=50)
axes[1].set_title("True Labels (PCA projection)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative Clustering
from scipy.cluster.hierarchy import dendrogram, linkage

agg = AgglomerativeClustering(n_clusters=5, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)

agg_silhouette = silhouette_score(X_scaled, agg_labels)
agg_ari = adjusted_rand_score(y_encoded, agg_labels)
agg_nmi = normalized_mutual_info_score(y_encoded, agg_labels)

print("Agglomerative Clustering (k=5, ward linkage):")
print(f"  Silhouette Score: {agg_silhouette:.4f}")
print(f"  Adjusted Rand Index (ARI): {agg_ari:.4f}")
print(f"  Normalized Mutual Information (NMI): {agg_nmi:.4f}")

# Dendrogram
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=20, leaf_font_size=10)
plt.title("Agglomerative Clustering — Dendrogram (Ward linkage, top 20 clusters)")
plt.xlabel("Cluster size")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# Clustering comparison summary
print("=" * 60)
print("CLUSTERING COMPARISON")
print("=" * 60)
print(f"{'Method':<30} {'Silhouette':>10} {'ARI':>10} {'NMI':>10}")
print("-" * 60)
print(f"{'K-Means (k=5)':<30} {kmeans_silhouette:>10.4f} {kmeans_ari:>10.4f} {kmeans_nmi:>10.4f}")
print(f"{'Agglomerative (k=5, ward)':<30} {agg_silhouette:>10.4f} {agg_ari:>10.4f} {agg_nmi:>10.4f}")
print("=" * 60)

### 3.5 - Deep Learning (MLP)

In [ ]:
# MLP Classifier (Deep Learning with sklearn)
mlp_params = {
    "hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32)],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate": ["adaptive"],
    "max_iter": [500],
}

mlp_grid = GridSearchCV(
    MLPClassifier(random_state=42, early_stopping=True, validation_fraction=0.15),
    mlp_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
mlp_grid.fit(X_scaled, y)

print("MLP — Best parameters:", mlp_grid.best_params_)
print(f"MLP — Best F1 macro (CV): {mlp_grid.best_score_:.4f}")

mlp_best = mlp_grid.best_estimator_
mlp_cv_scores = cross_val_score(mlp_best, X_scaled, y, cv=cv, scoring="f1_macro")
print(f"MLP — F1 macro per fold: {mlp_cv_scores}")
print(f"MLP — Mean F1 macro: {mlp_cv_scores.mean():.4f} (+/- {mlp_cv_scores.std():.4f})")

In [ ]:
# MLP — Classification report and confusion matrix
y_pred_mlp = cross_val_predict(mlp_best, X_scaled, y, cv=cv)

print("MLP — Classification Report (CV predictions):\n")
print(classification_report(y, y_pred_mlp))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y, y_pred_mlp, ax=ax, cmap="Purples")
ax.set_title("MLP — Confusion Matrix (Cross-Validated)")
plt.tight_layout()
plt.show()

### 3.6 - Model Comparison Summary

In [ ]:
# Summary of all supervised methods
results = pd.DataFrame({
    "Method": ["SVM (RBF)", "KNN", "Random Forest", "MLP (Deep Learning)"],
    "F1 Macro (mean)": [
        svm_cv_scores.mean(),
        knn_cv_scores.mean(),
        rf_cv_scores.mean(),
        mlp_cv_scores.mean(),
    ],
    "F1 Macro (std)": [
        svm_cv_scores.std(),
        knn_cv_scores.std(),
        rf_cv_scores.std(),
        mlp_cv_scores.std(),
    ],
})
results = results.sort_values("F1 Macro (mean)", ascending=False).reset_index(drop=True)

print("=" * 60)
print("SUPERVISED METHODS COMPARISON (5-fold Stratified CV)")
print("=" * 60)
print(results.to_string(index=False))
print("=" * 60)

# Bar plot comparison
plt.figure(figsize=(10, 5))
colors = ["steelblue", "seagreen", "darkorange", "mediumpurple"]
plt.bar(results["Method"], results["F1 Macro (mean)"], 
        yerr=results["F1 Macro (std)"], capsize=5, color=colors[:len(results)])
plt.ylim(0, 1)
plt.ylabel("F1 Macro Score")
plt.title("Model Comparison — F1 Macro (5-fold Stratified CV)")
plt.tight_layout()
plt.show()

### 3.7 - Prediction on Test Set

In [ ]:
# Extract features for test images (251-347)
TEST_DIR = Path("test")
TEST_IMG_DIR = TEST_DIR
TEST_MASK_DIR = TEST_DIR / "masks"

test_features = []
test_skipped = []

for i in range(251, 348):
    img_path = TEST_IMG_DIR / f"{i}.JPG"
    mask_path = TEST_MASK_DIR / f"binary_{i}.tif"

    if not img_path.exists() or not mask_path.exists():
        test_skipped.append(i)
        continue

    image = load_image(img_path)
    mask = load_mask(mask_path, image.size)
    image = np.array(image)

    row = {
        "ID": i,
        "bug_ratio": bug_ratio(mask),
        "shape_symmetry": shape_symmetry(mask),
        "color_symmetry": color_symmetry(image, mask),
    }
    row.update(rgb_features(image, mask))
    row.update(extra_shape_features(mask))
    test_features.append(row)

test_df = pd.DataFrame(test_features)
print(f"Test images processed: {len(test_df)}")
if test_skipped:
    print(f"Skipped: {test_skipped}")

In [ ]:
# Train best model on full training set and predict test set
# Select the best performing model from the comparison
best_method_name = results.iloc[0]["Method"]
print(f"Best method: {best_method_name}")

# Re-train on full training data with the best model's parameters
best_models = {
    "SVM (RBF)": svm_best,
    "KNN": knn_best,
    "Random Forest": rf_best,
    "MLP (Deep Learning)": mlp_best,
}
final_model = best_models[best_method_name]
final_model.fit(X_scaled, y)

# Scale test features with the same scaler
X_test = test_df[feature_cols].values
X_test_scaled = scaler.transform(X_test)

# Predict
y_test_pred = final_model.predict(X_test_scaled)

# Build results CSV
results_csv = pd.DataFrame({
    "ID": test_df["ID"].astype(int),
    "bug type": y_test_pred,
})

results_csv.to_csv("results.csv", index=False)
print(f"\nPredictions saved to results.csv ({len(results_csv)} entries)")
print(f"\nPrediction distribution:")
print(results_csv["bug type"].value_counts())
results_csv.head(10)